In [1]:
import pandas as pd
import re
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

STOPWORDS = set(stopwords.words('english'))

## Load and prepare data

In [2]:
df = pd.read_csv('../data/LabeledText.csv', encoding='latin-1')
df = df[['Caption', 'LABEL']].copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

df['clean'] = df['Caption'].apply(clean_text)

X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['LABEL'],
    test_size=0.2,
    random_state=42,
    stratify=df['LABEL']
)

## Train Naive Bayes

In [3]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred))

joblib.dump(pipeline, '../models/text/naive_bayes.pkl')

Accuracy: 0.672

              precision    recall  f1-score   support

    negative       0.74      0.58      0.65       291
     neutral       0.63      0.60      0.61       354
    positive       0.68      0.82      0.74       329

    accuracy                           0.67       974
   macro avg       0.68      0.67      0.67       974
weighted avg       0.68      0.67      0.67       974



['../models/text/naive_bayes.pkl']